# School Performance and Deprivation

## Problem Definition and Objectives

The objective of this project is to investigate how regional deprivation affects school performance across England. By analysing Key Stage 4 (GCSE level) results alongside the 2019 Indices of Multiple Deprivation (IMD), we aim to identify the specific socio-economic factors that have the strongest impact on educational outcomes.

### Core Research Questions
1. Performance by Deprivation Level Over Time: How does overall school performance (Attainment 8 / Progress 8) vary between the most and least deprived Local Authorities, and has this attainment gap widened or narrowed between 2019 and 2025?

2. Specific Deprivation Factors: Among the different sub-domains of deprivation (Income, Employment, Health, Crime, Living Environment, Barriers to Housing), which specific factor has the strongest negative correlation with a region's educational outcomes?

3. Subject-Specific Resilience: Are there certain core EBacc subjects (e.g., Maths, Science) that are more resilient to the effects of deprivation compared to others (e.g., Humanities, Languages)?

4. The Gender Gap and Deprivation: How does the performance gap between boys and girls change across different levels of regional deprivation?

5. Language Background and Deprivation: How does the performance of pupils with English as an Additional Language (EAL) compare to first-language English speakers across different deprivation levels?

Machine Learning Questions:

6. Can You Predict a School's Progress 8 Band from Non-Academic Data Alone?

7. What do "over-performing against deprivation" schools have in common — size, type, region, absence rates?



In [20]:
# Core library imports
import os
import requests
import pandas as pd
import numpy as np

## Data Sourcing

Before we load the data, we will download it directly from the official UK Government sources if it is not already present in our directory. This ensures the notebook is fully reproducible.

Rather than downloading a massive 226MB ZIP file containing dozens of unrelated tables, we download the two relevant datasets directly:

| Dataset | Source | Format |
|---------|--------|--------|
| IMD 2019 (File 7) | Ministry of Housing, Communities & Local Government | Direct CSV |
| KS4 Characteristics & Geography (2019-2025) | Department for Education — Explore Education Statistics API | Direct CSV |

In [21]:
# Create the data folder if it doesn't exist
os.makedirs("data", exist_ok=True)

# 1. English Indices of Deprivation 2019 (File 7 - LSOA scores, deciles, and denominators)
IoD_URL = "https://assets.publishing.service.gov.uk/media/5dc407b440f0b6379a7acc8d/File_7_-_All_IoD2019_Scores__Ranks__Deciles_and_Population_Denominators_3.csv"
IoD_PATH = "data/File_7_IoD2019.csv"

# 2. DfE KS4 Pupil Characteristics and Geography dataset (2018/19 up to 2024/25)
KS4_URL = "https://api.education.gov.uk/statistics/v1/data-sets/b3e19901-5d2b-b676-bb4c-e60937d74725/csv"
KS4_PATH = "data/ks4_characteristics_geography.csv"

# Downloading IoD 2019 dataset
if not os.path.exists(IoD_PATH):
    print("Downloading IoD 2019 dataset (File 7)...")
    response = requests.get(IoD_URL)
    response.raise_for_status()
    with open(IoD_PATH, "wb") as f:
        f.write(response.content)
    print("Saved data/File_7_IoD2019.csv")
else:
    print("data/File_7_IoD2019.csv already exists.")

# Downloading KS4 Characteristics & Geography dataset
if not os.path.exists(KS4_PATH):
    print("Downloading KS4 Characteristics & Geography dataset...")
    response = requests.get(KS4_URL)
    response.raise_for_status()
    with open(KS4_PATH, "wb") as f:
        f.write(response.content)
    print("Saved data/ks4_characteristics_geography.csv")
else:
    print("data/ks4_characteristics_geography.csv already exists.")

data/File_7_IoD2019.csv already exists.
data/ks4_characteristics_geography.csv already exists.


## Data Loading and Exploration

In this section, we will load our primary datasets:
1. **Indices of Multiple Deprivation (IMD) 2019:** Provides deprivation deciles and sub-domain scores for each Local Authority.
2. **DfE Key Stage 4 Pupil Characteristics and Geography:** Provides performance breakdowns by sex, language, and disadvantage status at Local Authority level from 2018/19 up to 2024/25.

In [22]:
# Load IoD dataset
iod_data = pd.read_csv(IoD_PATH, low_memory=False) # low_memory=False to avoid dtype warnings for large files

# Load KS4 Pupil Characteristics and Geography dataset
performance_data = pd.read_csv(KS4_PATH, low_memory=False) # low_memory=False to avoid dtype warnings for large files

print(f"IoD shape: {iod_data.shape}")
print(f"KS4 shape: {performance_data.shape}")

IoD shape: (32844, 57)
KS4 shape: (342410, 200)


## Initial Data Inspection

Now that the data is loaded, let's perform an initial inspection to understand the structure, check the column names, and look for any immediate data quality issues.

In [23]:
print("--- IoD 2019 Dataset ---")
iod_data.head()

--- IoD 2019 Dataset ---


,LSOA code (2011),LSOA name (2011),Local Authority District code (2019),Local Authority District name (2019),Index of Multiple Deprivation (IMD) Score,Index of Multiple Deprivation (IMD) Rank (where 1 is most deprived),Index of Multiple Deprivation (IMD) Decile (where 1 is most deprived 10% of LSOAs),Income Score (rate),Income Rank (where 1 is most deprived),Income Decile (where 1 is most deprived 10% of LSOAs),...,Indoors Sub-domain Rank (where 1 is most deprived),Indoors Sub-domain Decile (where 1 is most deprived 10% of LSOAs),Outdoors Sub-domain Score,Outdoors Sub-domain Rank (where 1 is most deprived),Outdoors Sub-domain Decile (where 1 is most deprived 10% of LSOAs),Total population: mid 2015 (excluding prisoners),Dependent Children aged 0-15: mid 2015 (excluding prisoners),Population aged 16-59: mid 2015 (excluding prisoners),Older population aged 60 and over: mid 2015 (excluding prisoners),Working age population 18-59/64: for use with Employment Deprivation Domain (excluding prisoners)
0,E01000001,City of London 001A,E09000001,City of London,6.208,29199,9,0.007,32831,10,...,16364,5,1.503,1615,1,1296,175,656,465,715
1,E01000002,City of London 001B,E09000001,City of London,5.143,30379,10,0.034,29901,10,...,22676,7,1.196,2969,1,1156,182,580,394,620
2,E01000003,City of London 001C,E09000001,City of London,19.402,14915,5,0.086,18510,6,...,17318,6,2.207,162,1,1350,146,759,445,804
3,E01000005,City of London 001E,E09000001,City of London,28.652,8678,3,0.211,6029,2,...,25218,8,1.769,849,1,1121,229,692,200,683
4,E01000006,Barking and Dagenham 016A,E09000002,Barking and Dagenham,19.837,14486,5,0.117,14023,5,...,14745,5,0.969,4368,2,2040,522,1297,221,1285


In [24]:
print("\n---IoD 2019 Info---")
print(iod_data.info())


---IoD 2019 Info---
<class 'pandas.DataFrame'>
RangeIndex: 32844 entries, 0 to 32843
Data columns (total 57 columns):
 #   Column                                                                                              Non-Null Count  Dtype  
---  ------                                                                                              --------------  -----  
 0   LSOA code (2011)                                                                                    32844 non-null  str    
 1   LSOA name (2011)                                                                                    32844 non-null  str    
 2   Local Authority District code (2019)                                                                32844 non-null  str    
 3   Local Authority District name (2019)                                                                32844 non-null  str    
 4   Index of Multiple Deprivation (IMD) Score                                                           32

In [32]:

print("\n--- Performance Dataset ---")
performance_data.head()


--- Performance Dataset ---


,time_period,time_identifier,geographic_level,country_code,country_name,region_code,region_name,old_la_code,new_la_code,la_name,...,valueaddedlan_sum,valueaddedlan_average,valueaddedlan_lower_95_ci,valueaddedlan_upper_95_ci,qual_entries_sum,qual_entries_average,gcse_entries_sum,gcse_entries_average,qual_entries_without_discounting_sum,qual_entries_without_discounting_average
0,202425,Academic year,National,E92000001,England,NaN,NaN,NaN,NaN,NaN,...,z,z,z,z,4901678.49,7.8,4525185,7.2,4942447,7.9
1,202425,Academic year,National,E92000001,England,NaN,NaN,NaN,NaN,NaN,...,z,z,z,z,2485577.91,7.8,2286227,7.1,2506494,7.8
2,202425,Academic year,National,E92000001,England,NaN,NaN,NaN,NaN,NaN,...,z,z,z,z,2416100.58,7.9,2238958,7.3,2435953,8
3,202425,Academic year,National,E92000001,England,NaN,NaN,NaN,NaN,NaN,...,z,z,z,z,253719.3,8,235518,7.5,254870,8.1
4,202425,Academic year,National,E92000001,England,NaN,NaN,NaN,NaN,NaN,...,z,z,z,z,123521.17,7.9,113991,7.3,124166,7.9


In [33]:
print("\n--- Performance Info ---")
print(performance_data.info())


--- Performance Info ---
<class 'pandas.DataFrame'>
RangeIndex: 342410 entries, 0 to 342409
Columns: 200 entries, time_period to qual_entries_without_discounting_average
dtypes: float64(10), int64(4), str(186)
memory usage: 522.5 MB
None


In [34]:
# list of columns in datasets
print("\n--- IoD Columns ---")
print(iod_data.columns.tolist()) # Convert columns to list for better readability

print("\n--- Performance Columns ---")
print(performance_data.columns.tolist()) # Convert columns to list for better readability


--- IoD Columns ---
['LSOA code (2011)', 'LSOA name (2011)', 'Local Authority District code (2019)', 'Local Authority District name (2019)', 'Index of Multiple Deprivation (IMD) Score', 'Index of Multiple Deprivation (IMD) Rank (where 1 is most deprived)', 'Index of Multiple Deprivation (IMD) Decile (where 1 is most deprived 10% of LSOAs)', 'Income Score (rate)', 'Income Rank (where 1 is most deprived)', 'Income Decile (where 1 is most deprived 10% of LSOAs)', 'Employment Score (rate)', 'Employment Rank (where 1 is most deprived)', 'Employment Decile (where 1 is most deprived 10% of LSOAs)', 'Education, Skills and Training Score', 'Education, Skills and Training Rank (where 1 is most deprived)', 'Education, Skills and Training Decile (where 1 is most deprived 10% of LSOAs)', 'Health Deprivation and Disability Score', 'Health Deprivation and Disability Rank (where 1 is most deprived)', 'Health Deprivation and Disability Decile (where 1 is most deprived 10% of LSOAs)', 'Crime Score', 'C

In [35]:
#distinct values in geographic_level column of performance dataset
print("\n--- Geographic Level Distinct Values ---")
print(performance_data['geographic_level'].unique())


--- Geographic Level Distinct Values ---
<StringArray>
['National', 'Regional', 'Local authority']
Length: 3, dtype: str


In [13]:
#statistics for IoD and KS4 datasets
print("\n--- IoD 2019 Dataset Statistics ---")
print(iod_data.describe())
print("\n--- KS4 Dataset Statistics ---")
print(performance_data.describe())


--- IoD 2019 Dataset Statistics ---
       Index of Multiple Deprivation (IMD) Score  \
count                               32844.000000   
mean                                   21.669393   
std                                    15.332229   
min                                     0.541000   
25%                                     9.913750   
50%                                    17.647500   
75%                                    29.583000   
max                                    92.735000   

       Index of Multiple Deprivation (IMD) Rank (where 1 is most deprived)  \
count                                       32844.000000                     
mean                                        16422.499208                     
std                                          9481.390454                     
min                                             1.000000                     
25%                                          8211.750000                     
50%                       

## 3. Data Selection and Feature Engineering

Based on our research questions, we need to narrow down the hundreds of columns in these datasets. Below is our mapping of the specific columns we need for our analysis:

## Data Selection – Columns to Retain

Based on our research questions, we narrow down from ~200 performance columns and 57 deprivation columns to only those listed below.

--- Performance Columns ---
'time_period', 'geographic_level', 'region_code', 'region_name', 'old_la_code', 'new_la_code', 'la_name', 'sex', 'ethnicity_major', 'ethnicity_minor', 'disadvantage_status', 'first_language', 'school_admission_type', 'pupil_count', 'attainment8_average', 'progress8_average', 'progress8eng_average', 'progress8mat_average', 'progress8ebacc_average', 'progress8open_average', 'gcse_entering_total', 'ebacceng_aps_average', 'ebaccmat_aps_average', 'ebaccsci_aps_average', 'ebacchum_aps_average', 'ebacclan_aps_average', 'attainment8eng_average', 'attainment8mat_average', 'attainment8ebacc_average', 'attainment8open_average', 'attainment8open_gcse_average', 'attainment8open_nongcse_average', 'attainment8ebacc_slotfill_average', 'attainment8open_slotfill_average', 'valueaddedsci_average', 'valueaddedhum_average', 'valueaddedlan_average', 'qual_entries_average', 'gcse_entries_average'

# COLUMNS TO RETAIN IN PERFORMANCE TABLE
Identifiers and filters
- new_la_code/la_name - for joining to deprivation dataset
- school_urn/school_name - unique school id
- establishment_type_group - academy, independent, maintained
- sex
- disadvantage_status
- first_language
- breakdown_topic


Outcome measures
- attainment8_average
- progress8_average
- progress8_lower_95_ci / progress8_upper_95_ci
- ebacc_aps_average

Subject-specific (for Q3)
- attainment8eng_average, attainment8mat_average, attainment8ebacc_average
- progress8eng_average, progress8mat_average, progress8ebacc_average, progress8open_average
- ebacceng_95_percent, ebaccmat_95_percent, ebaccsci_95_percent, ebacchum_95_percent, ebacclan_95_percent

Pupil characteristics
- pupil_count


# COLUMNS TO RETAIN IN DEPRIVATION DATA
--- IoD Columns ---
'Local Authority District code (2019)', 'Local Authority District name (2019)', 'Index of Multiple Deprivation (IMD) Score', 'Income Score (rate)', 'Employment Score (rate)', 'Education, Skills and Training Score', 'Health Deprivation and Disability Score', 'Crime Score', 'Barriers to Housing and Services Score', 'Living Environment Score', 'Income Deprivation Affecting Children Index (IDACI) Score (rate)', 'Children and Young People Sub-domain Score', 'Geographical Barriers Sub-domain Score', 'Wider Barriers Sub-domain Score', 'Indoors Sub-domain Score', 'Outdoors Sub-domain Score'

Identifier
- Local Authority District code (2019)

Deprivation (for Q1)
- Index of Multiple Deprivation (IMD) Score

Specific Deprivation Factors (for Q2)
- Income Score (rate)
- Employment Score (rate)
- Health Deprivation and Disability Score
- Crime Score
- Barriers to Housing and Services Score
- Living Environment Score
- Income Deprivation Affecting Children Index (IDACI) Score (rate)
- Children and Young People Sub-domain Score

'''








### Filtering the Performance Data

The KS4 dataset has multiple rows per school depending on the demographic breakdown. To make analysis easier, we will split this into separate DataFrames for our specific research questions (Overall, Gender gap, and EAL).

In [17]:
# Filtering performance rows (several per school, so trimming down) & columns
performance_total = performance_data[
    (performance_data['breakdown_topic'] == 'Total') &
    (performance_data['sex'] == 'Total') &
    (performance_data['disadvantage_status'] == 'Total') &
    (performance_data['first_language'] == 'Total')
][['time_period', 'geographic_level', 'ebacceng_aps_average', 'ebaccmat_aps_average', 'ebaccsci_aps_average', 'ebacchum_aps_average', 'ebacclan_aps_average',  
   'establishment_type_group', 'pupil_count',
   'attainment8_average', 'progress8_average', 
   'ebacc_aps_average']].copy()


# For Q4 (gender gap) - use sex breakdown
perf_gender = performance_data[
    (performance_data['breakdown_topic'] == 'Sex') &
    (performance_data['disadvantage_status'] == 'Total') &
    (performance_data['first_language'] == 'Total')
][['time_period', 'geographic_level', 'ebacceng_aps_average', 'ebaccmat_aps_average', 'ebaccsci_aps_average', 'ebacchum_aps_average', 'ebacclan_aps_average', 
   'establishment_type_group', 'pupil_count',
   'attainment8_average', 'progress8_average', 
   'ebacc_aps_average']].copy()

# For Q5 (EAL) - use first language breakdown
perf_eal = performance_data[
    (performance_data['breakdown_topic'] == 'First language')
][['time_period', 'geographic_level', 'ebacceng_aps_average', 'ebaccmat_aps_average', 'ebaccsci_aps_average', 'ebacchum_aps_average', 'ebacclan_aps_average',
   'establishment_type_group', 'pupil_count',
   'attainment8_average', 'progress8_average', 
   'ebacc_aps_average']].copy()

print("---NEW PERFORMANCE DATA STRUCTURE ------")
print(performance_total.head())
print("\nInfo:")
print(performance_total.info())

---NEW PERFORMANCE DATA STRUCTURE ------
      time_period geographic_level ebacceng_aps_average ebaccmat_aps_average  \
0          202425         National                 4.89                 4.55   
2773       202425         Regional                 4.75                 4.35   
3051       202425         Regional                 4.78                 4.37   
3342       202425         Regional                 4.76                 4.39   
3629       202425         Regional                 4.79                 4.49   

     ebaccsci_aps_average ebacchum_aps_average ebacclan_aps_average  \
0                    4.47                 3.75                 2.38   
2773                 4.24                 3.59                 1.96   
3051                 4.29                 3.53                 2.18   
3342                 4.29                 3.63                 2.06   
3629                  4.4                 3.71                 2.14   

     establishment_type_group  pupil_count attainme

### Aggregating Deprivation Data (LSOA to Local Authority)

The Indices of Deprivation dataset is at the granular LSOA level. To merge it with our KS4 data, we need to aggregate these scores up to the Local Authority level by taking the mean.

In [18]:
iod_la = iod_data.groupby(
    ['Local Authority District code (2019)', 'Local Authority District name (2019)']
).agg(
    imd_score          = ('Index of Multiple Deprivation (IMD) Score', 'mean'),
    income_score       = ('Income Score (rate)', 'mean'),
    employment_score   = ('Employment Score (rate)', 'mean'),
    health_score       = ('Health Deprivation and Disability Score', 'mean'),
    crime_score        = ('Crime Score', 'mean'),
    barriers_score     = ('Barriers to Housing and Services Score', 'mean'),
    living_env_score   = ('Living Environment Score', 'mean'),
    idaci_score        = ('Income Deprivation Affecting Children Index (IDACI) Score (rate)', 'mean'),
    children_subdomain = ('Children and Young People Sub-domain Score', 'mean'),
).reset_index()

iod_la.rename(columns={
    'Local Authority District code (2019)': 'la_code'
}, inplace=True)

print("---NEW DEPRIVATION DATA STRUCTURE ------")
print(iod_la.head())
print("\nInfo:")
print(iod_la.info())

---NEW DEPRIVATION DATA STRUCTURE ------
     la_code Local Authority District name (2019)  imd_score  income_score  \
0  E06000001                           Hartlepool  34.853121      0.226241   
1  E06000002                        Middlesbrough  40.443116      0.251326   
2  E06000003                 Redcar and Cleveland  29.841330      0.185534   
3  E06000004                     Stockton-on-Tees  25.244108      0.161017   
4  E06000005                           Darlington  26.787000      0.160000   

   employment_score  health_score  crime_score  barriers_score  \
0          0.182017      0.874190     0.582345       13.764345   
1          0.190605      1.192070     0.680523       14.206791   
2          0.162909      0.807034     0.152227       14.020648   
3          0.132700      0.670000    -0.137200       15.717142   
4          0.131477      0.582631     0.579523       11.438292   

   living_env_score  idaci_score  children_subdomain  
0          7.298431     0.266483      

## 4. Missing Data and Suppressions Analysis

The DfE uses the letters `'c'` and `'z'` to indicate suppressed data (due to small cohort sizes or COVID-affected years). We need to check how prevalent these suppressions are before doing any numerical conversions.

In [19]:
# MISSING DATA - PERFORMANCE
print("--- MISSING DATA INITIAL ANALYSIS---")
# Per column counts
c_by_col = (performance_total == 'c').sum()
z_by_col = (performance_total == 'z').sum()

# Show only columns that actually have them
performance_total_suppression_summary = pd.DataFrame({
    'c_count': c_by_col,
    'z_count': z_by_col
})
performance_total_suppression_summary = performance_total_suppression_summary[
    (performance_total_suppression_summary['c_count'] > 0) | 
    (performance_total_suppression_summary['z_count'] > 0)
].sort_values('c_count', ascending=False)

print(f"\nPerformance total supression summary: \n {performance_total_suppression_summary}")


# Gender 
# Per column counts
c_by_col = (perf_gender == 'c').sum()
z_by_col = (perf_gender == 'z').sum()

# Show only columns that actually have them
perf_gender_suppression_summary = pd.DataFrame({
    'c_count': c_by_col,
    'z_count': z_by_col
})
perf_gender_suppression_summary = perf_gender_suppression_summary[
    (perf_gender_suppression_summary['c_count'] > 0) | 
    (perf_gender_suppression_summary['z_count'] > 0)
].sort_values('c_count', ascending=False)

print(f"\nPerformance gender supression summary: \n {perf_gender_suppression_summary}")

# EAL
# Per column counts
c_by_col = (perf_eal == 'c').sum()
z_by_col = (perf_eal == 'z').sum()

# Show only columns that actually have them
perf_eal_suppression_summary = pd.DataFrame({
    'c_count': c_by_col,
    'z_count': z_by_col
})
perf_eal_suppression_summary = perf_eal_suppression_summary[
    (perf_eal_suppression_summary['c_count'] > 0) | 
    (perf_eal_suppression_summary['z_count'] > 0)
].sort_values('c_count', ascending=False)

print(f"\nPerformance EAL supression summary: \n {perf_eal_suppression_summary}")

# MISSING DATA - IOD
print(f"\nIOD Missing data summary: {iod_data.isnull().sum().sum()} total missing values")

--- MISSING DATA INITIAL ANALYSIS---

Performance total supression summary: 
                       c_count  z_count
ebacceng_aps_average        0        9
ebaccmat_aps_average        0        9
ebaccsci_aps_average        0        9
ebacchum_aps_average        0        9
ebacclan_aps_average        0        9
attainment8_average         0        5
progress8_average           0      498
ebacc_aps_average           0        8

Performance gender supression summary: 
                    c_count  z_count
progress8_average        0      978

Performance EAL supression summary: 
                    c_count  z_count
progress8_average        0      975

IOD Missing data summary: 0 total missing values
